In [ ]:
!pip install -q sentence-transformers

from sentence_transformers import CrossEncoder

reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

def rerank(query, docs):
    pairs = [[query, doc] for doc in docs]

    scores = reranker.predict(pairs)

    scored_docs = list(zip(docs, scores))

    scored_docs = sorted(scored_docs, key=lambda x: x[1], reverse=True)

    return [doc for doc, score in scored_docs]




def mrr(search_fn, num_samples=150):
    total = 0

    for i in range(num_samples):
        query = queries[i]
        true_context = dataset[i]['context']

        results = search_fn(query)

        for rank, doc in enumerate(results):
            if doc == true_context:
                total += 1 / (rank + 1)
                break

    return total / num_samples

def search_with_rerank(query, top_k=5):
    candidate_docs = dense_search(query, top_k=top_k)
    reranked_docs = rerank(query, candidate_docs)
    return reranked_docs

dense_mrr = mrr(dense_search)
rerank_mrr = mrr(search_with_rerank)

print("Dense MRR:", dense_mrr)
print("Dense + Rerank MRR:", rerank_mrr)


def bm25_with_rerank(query, top_k=5):
    candidate_docs = bm25_search(query, top_k=top_k)
    reranked_docs = rerank(query, candidate_docs)
    return reranked_docs

bm25_mrr = mrr(bm25_search)
bm25_rerank_mrr = mrr(bm25_with_rerank)

print("BM25 MRR:", bm25_mrr)
print("BM25 + Rerank MRR:", bm25_rerank_mrr)